# Advanced Titanic Competition Pipeline

This notebook implements a competition-grade machine learning pipeline for improving Kaggle leaderboard performance.

Techniques Used:

- Advanced Feature Engineering
- One-Hot Encoding
- Stratified K-Fold Cross Validation
- CatBoost
- XGBoost
- LightGBM
- Ensemble Blending

Goal:
Improve leaderboard score toward 0.85+

In [2]:
! pip install catboost

  Using cached catboost-1.2.10-cp313-cp313-win_amd64.whl.metadata (1.5 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
  Using cached plotly-6.7.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached narwhals-2.21.0-py3-none-any.whl.metadata (16 kB)
Using cached catboost-1.2.10-cp313-cp313-win_amd64.whl (100.2 MB)
Using cached graphviz-0.21-py3-none-any.whl (47 kB)
Using cached plotly-6.7.0-py3-none-any.whl (9.9 MB)
Using cached narwhals-2.21.0-py3-none-any.whl (451 kB)



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# ============================================
# IMPORT LIBRARIES
# ============================================

import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

from sklearn.metrics import accuracy_score

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier
)

from xgboost import XGBClassifier

from lightgbm import LGBMClassifier

from catboost import CatBoostClassifier

ModuleNotFoundError: No module named 'catboost'

In [ ]:
# ============================================
# LOAD DATASET
# ============================================

train_df = pd.read_csv(
    "titanic/train.csv"
)

test_df = pd.read_csv(
    "titanic/test.csv"
)

In [ ]:
# ============================================
# SAVE PASSENGER IDs
# ============================================

test_passenger_ids = test_df["PassengerId"]

In [ ]:
# ============================================
# COMBINE DATASETS
# ============================================

full_df = pd.concat(
    [train_df, test_df],
    axis=0,
    ignore_index=True
)

In [ ]:
# ============================================
# HANDLE MISSING VALUES
# ============================================

full_df["Age"] = full_df["Age"].fillna(
    full_df["Age"].median()
)

full_df["Fare"] = full_df["Fare"].fillna(
    full_df["Fare"].median()
)

full_df["Embarked"] = full_df["Embarked"].fillna(
    full_df["Embarked"].mode()[0]
)

full_df["Cabin"] = full_df["Cabin"].fillna(
    "Unknown"
)

In [ ]:
# ============================================
# FAMILY SIZE
# ============================================

full_df["FamilySize"] = (
    full_df["SibSp"] +
    full_df["Parch"] + 1
)

In [ ]:
# ============================================
# FAMILY TYPE
# ============================================

def family_type(size):
    
    if size == 1:
        return "Alone"
    
    elif size <= 4:
        return "Small"
    
    else:
        return "Large"

full_df["FamilyType"] = full_df[
    "FamilySize"
].apply(family_type)

In [ ]:
# ============================================
# FARE PER PERSON
# ============================================

full_df["FarePerPerson"] = (
    full_df["Fare"] /
    full_df["FamilySize"]
)

In [ ]:
# ============================================
# CHILD FEATURE
# ============================================

full_df["Child"] = (
    full_df["Age"] < 16
).astype(int)

In [ ]:
# ============================================
# MOTHER FEATURE
# ============================================

full_df["Mother"] = (
    (full_df["Sex"] == "female") &
    (full_df["Parch"] > 0) &
    (full_df["Age"] > 18)
).astype(int)

In [ ]:
# ============================================
# CABIN KNOWN FEATURE
# ============================================

full_df["CabinKnown"] = (
    full_df["Cabin"] != "Unknown"
).astype(int)

In [ ]:
# ============================================
# DECK FEATURE
# ============================================

full_df["Deck"] = full_df[
    "Cabin"
].str[0]

full_df["Deck"] = full_df[
    "Deck"
].replace(
    ['G', 'T'],
    'Rare'
)

In [ ]:
# ============================================
# TICKET FREQUENCY
# ============================================

full_df["TicketFrequency"] = full_df.groupby(
    "Ticket"
)["Ticket"].transform("count")

In [ ]:
# ============================================
# TITLE FEATURE
# ============================================

full_df["Title"] = full_df["Name"].str.extract(
    ' ([A-Za-z]+)\.',
    expand=False
)

full_df["Title"] = full_df["Title"].replace(
    [
        'Lady','Countess','Capt','Col',
        'Don','Dr','Major','Rev',
        'Sir','Jonkheer','Dona'
    ],
    'Rare'
)

full_df["Title"] = full_df["Title"].replace(
    ['Mlle','Ms'],
    'Miss'
)

full_df["Title"] = full_df["Title"].replace(
    'Mme',
    'Mrs'
)

In [ ]:
# ============================================
# DROP UNUSED FEATURES
# ============================================

full_df.drop(
    
    columns=[
        "PassengerId",
        "Name",
        "Ticket",
        "Cabin"
    ],
    
    inplace=True
)

In [ ]:
# ============================================
# SPLIT DATASETS
# ============================================

train_processed = full_df[
    full_df["Survived"].notnull()
]

test_processed = full_df[
    full_df["Survived"].isnull()
]

In [ ]:
# ============================================
# INPUT AND TARGET
# ============================================

X = train_processed.drop(
    "Survived",
    axis=1
)

y = train_processed["Survived"]

X_test = test_processed.drop(
    "Survived",
    axis=1
)

In [ ]:
# ============================================
# FEATURE TYPES
# ============================================

categorical_features = X.select_dtypes(
    include=["object"]
).columns

numerical_features = X.select_dtypes(
    exclude=["object"]
).columns

In [ ]:
# ============================================
# NUMERICAL PIPELINE
# ============================================

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [ ]:
# ============================================
# CATEGORICAL PIPELINE
# ============================================

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [ ]:
# ============================================
# COLUMN TRANSFORMER
# ============================================

preprocessor = ColumnTransformer(
    
    transformers=[
        
        (
            "num",
            numeric_transformer,
            numerical_features
        ),
        
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [ ]:
cat_model = CatBoostClassifier(
    
    iterations=2000,
    
    learning_rate=0.01,
    
    depth=5,
    
    loss_function='Logloss',
    
    verbose=0,
    
    random_state=42
)

In [ ]:
xgb_model = XGBClassifier(
    
    n_estimators=1000,
    
    learning_rate=0.02,
    
    max_depth=3,
    
    subsample=0.8,
    
    colsample_bytree=0.8,
    
    random_state=42
)

In [ ]:
lgbm_model = LGBMClassifier(
    
    n_estimators=1000,
    
    learning_rate=0.02,
    
    max_depth=3,
    
    random_state=42
)

In [ ]:
# ============================================
# FULL MODEL PIPELINES
# ============================================

cat_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", cat_model)
    ]
)

xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", xgb_model)
    ]
)

lgbm_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", lgbm_model)
    ]
)

In [ ]:
# ============================================
# STRATIFIED K FOLD
# ============================================

skf = StratifiedKFold(
    
    n_splits=5,
    
    shuffle=True,
    
    random_state=42
)

In [ ]:
cat_scores = cross_val_score(
    
    cat_pipeline,
    
    X,
    
    y,
    
    cv=skf,
    
    scoring="accuracy"
)

print("CatBoost Accuracy:")
print(cat_scores.mean())

In [ ]:
xgb_scores = cross_val_score(
    
    xgb_pipeline,
    
    X,
    
    y,
    
    cv=skf,
    
    scoring="accuracy"
)

print("XGBoost Accuracy:")
print(xgb_scores.mean())

In [ ]:
lgbm_scores = cross_val_score(
    
    lgbm_pipeline,
    
    X,
    
    y,
    
    cv=skf,
    
    scoring="accuracy"
)

print("LightGBM Accuracy:")
print(lgbm_scores.mean())

In [ ]:
# ============================================
# TRAIN MODELS
# ============================================

cat_pipeline.fit(X, y)

xgb_pipeline.fit(X, y)

lgbm_pipeline.fit(X, y)

In [ ]:
# ============================================
# PREDICTION PROBABILITIES
# ============================================

cat_prob = cat_pipeline.predict_proba(
    X_test
)[:,1]

xgb_prob = xgb_pipeline.predict_proba(
    X_test
)[:,1]

lgbm_prob = lgbm_pipeline.predict_proba(
    X_test
)[:,1]

In [ ]:
# ============================================
# BLEND PREDICTIONS
# ============================================

final_prob = (
    
    0.4 * cat_prob +
    
    0.3 * xgb_prob +
    
    0.3 * lgbm_prob
)

In [ ]:
# ============================================
# FINAL PREDICTIONS
# ============================================

final_predictions = (
    final_prob >= 0.5
).astype(int)

In [ ]:
# ============================================
# CREATE SUBMISSION FILE
# ============================================

submission = pd.DataFrame({
    
    "PassengerId": test_passenger_ids,
    
    "Survived": final_predictions
})

In [ ]:
# ============================================
# SAVE SUBMISSION
# ============================================

submission.to_csv(
    "submission_advanced.csv",
    index=False
)

print("submission_advanced.csv created successfully!")